Import the libraries.

In [15]:
import numpy as np
import pandas as pd

import seaborn as sns
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

Read the dataset.

In [16]:
df = pd.read_csv('Datasets/titanic.csv',usecols=['Age','Pclass','SibSp','Parch','Survived'])
df.dropna()
df.head()

,Survived,Pclass,Age,SibSp,Parch
0,0,3,22.0,1,0
1,1,1,38.0,1,0
2,1,3,26.0,0,0
3,1,1,35.0,1,0
4,0,3,35.0,0,0


Drop nan/null values to apply logestic regression.

In [17]:
x = df.dropna()[['Pclass', 'SibSp', 'Parch', 'Age']].copy()
y = df.dropna()['Survived'].copy()
x,y

(     Pclass  SibSp  Parch   Age
 0         3      1      0  22.0
 1         1      1      0  38.0
 2         3      0      0  26.0
 3         1      1      0  35.0
 4         3      0      0  35.0
 ..      ...    ...    ...   ...
 885       3      0      5  39.0
 886       2      0      0  27.0
 887       1      0      0  19.0
 889       1      0      0  26.0
 890       3      0      0  32.0
 
 [714 rows x 4 columns],
 0      0
 1      1
 2      1
 3      1
 4      0
       ..
 885    0
 886    0
 887    1
 889    1
 890    0
 Name: Survived, Length: 714, dtype: int64)

Cross validation score.

In [18]:
np.mean(cross_val_score(LogisticRegression(max_iter=1000),x,y,scoring='accuracy',cv=20))

np.float64(0.6933333333333332)

Feature construction.

In [19]:
x['family_size'] = x['SibSp'] + x['Parch'] + 1
x.head()

,Pclass,SibSp,Parch,Age,family_size
0,3,1,0,22.0,2
1,1,1,0,38.0,2
2,3,0,0,26.0,1
3,1,1,0,35.0,2
4,3,0,0,35.0,1


A function to find the type of family (alone, medium, large).

In [20]:
def family_type(num):
    if num == 1:
        return 0
    elif num > 1 and num < 5:
        return 1
    else:
        return 2
x['family_type'] = x['family_size'].apply(family_type)
x.head()

,Pclass,SibSp,Parch,Age,family_size,family_type
0,3,1,0,22.0,2,1
1,1,1,0,38.0,2,1
2,3,0,0,26.0,1,0
3,1,1,0,35.0,2,1
4,3,0,0,35.0,1,0


Remove other columns.

In [21]:
x = x.drop(['SibSp', 'Parch', 'family_size'], axis=1)

Cross validation score.

In [22]:
np.mean(cross_val_score(LogisticRegression(max_iter=1000),x,y,scoring='accuracy',cv=20))

np.float64(0.7003174603174602)

Feature Splitting.

In [23]:
df1 = pd.read_csv('Datasets/titanic.csv', usecols=['Name', 'Survived'])
df1.describe()

,Survived
count,891.000000
mean,0.383838
std,0.486592
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,1.000000


Extract the title of data.

In [24]:
df1['Title'] = df1['Name'].str.split(',', expand=True)[1].str.split('.', expand=True)[0].str.strip() # Expand=True is used to split the string into multiple columns. 
# df1['Title'] = df1['Name'].str.split(', ', expand=True)[1].str.split('.', expand=True)[0] same thing.
df1.head()

# Names are in the form as 'Braund, Mr. Owen Harris', we have to extract the title like mr., mrs. from the name. So we will split the name by ',' and then take the second part and then split it by '.' and take the first part and then strip it to remove any extra spaces. This will give us the title of the person. We can then use this title to create a new feature in our dataset.
# .str is used to access the string methods of the pandas series. We can use it to split the string and extract the required part.

,Survived,Name,Title
0,0,"Braund, Mr. Owen Harris",Mr
1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs
2,1,"Heikkinen, Miss. Laina",Miss
3,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs
4,0,"Allen, Mr. William Henry",Mr


Analysis on title.

In [25]:
(df1.groupby('Title')['Survived'].mean()).sort_values(ascending=False)

Title
Lady            1.000000
Ms              1.000000
Sir             1.000000
Mme             1.000000
the Countess    1.000000
Mlle            1.000000
Mrs             0.792000
Miss            0.697802
Master          0.575000
Major           0.500000
Col             0.500000
Dr              0.428571
Mr              0.156673
Capt            0.000000
Jonkheer        0.000000
Don             0.000000
Rev             0.000000
Name: Survived, dtype: float64

In [26]:
df1['Is_Married'] = 0
df1['Is_Married'].loc[df1['Title'] == 'Mrs'] = 1

/tmp/ipykernel_13071/3636290223.py:2: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df1['Is_Married'].loc[df1['Title'] == 'Mrs'] = 1
/tmp/ipykernel_13071/3636290223.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sl

In [27]:
df1.head()


,Survived,Name,Title,Is_Married
0,0,"Braund, Mr. Owen Harris",Mr,0
1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs,1
2,1,"Heikkinen, Miss. Laina",Miss,0
3,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs,1
4,0,"Allen, Mr. William Henry",Mr,0
